In [ ]:
import os
import sys
import yaml
import types
import torch

In [ ]:
sys.path.insert(0, "..")
from models.beta_vae import BetaVAE
from utils import make_run_tag
from data_utils import build_encode_dataloader

In [ ]:
REPO_ROOT = os.path.abspath("..")
CKPT_PATH = None # None → load latest checkpoint from experiment output dir
CONFIG_PATH = "config/betavae_full_stimuli_high_res.yaml"   # relative to repo root

## Load model & checkpoint

In [ ]:
with open(os.path.join(REPO_ROOT, CONFIG_PATH)) as f:
    cfg = types.SimpleNamespace(**yaml.safe_load(f))

cfg.output_dir = os.path.join(REPO_ROOT, f"{cfg.output_dir.lstrip('./')}_{make_run_tag(cfg)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BetaVAE(
    input_channels=cfg.input_channels,
    input_height=cfg.input_height,
    input_width=cfg.input_width,
    latent_dim=cfg.latent_dim,
    hidden_dim=cfg.hidden_dim,
    beta=cfg.beta,
    device=device,
)

if CKPT_PATH is None:
    ckpt_dir = os.path.join(cfg.output_dir, "checkpoints")
    CKPT_PATH = os.path.join(ckpt_dir, "best_ckpt.pt")
    if not os.path.exists(CKPT_PATH):
        raise FileNotFoundError(f"best_ckpt.pt not found in {ckpt_dir}")

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded: {CKPT_PATH}  (step {ckpt['step']})")

## Encode

In [ ]:
encode_data_path = "data/500Stimuli"
encode_dl = build_encode_dataloader(encode_data_path, image_size=cfg.input_height, batch_size=64)

In [ ]:
all_mu = []
with torch.no_grad():
    for batch in encode_dl:
        batch = batch.to(device)
        _, mu, _ = model(batch)
        all_mu.append(mu.cpu())

all_mu = torch.cat(all_mu, dim=0).numpy()  # [N, latent_dim]
print(f"Collected μ for {all_mu.shape[0]} images across {all_mu.shape[1]} latent dimensions")